# IRMAS Model Test (On-the-Fly Features)

This notebook evaluates one trained checkpoint on IRMAS test data without pre-generating test features.

It reads:
- test set location from `src/configs/audio_params.yaml` (`datasets.irmas.test`)
- feature extraction parameters from the `audio` config
- model/backbone/feature mode from the run checkpoint + `run_config.yaml`


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src.test.test import evaluate_irmas_part1_run

try:
    from sklearn.metrics import ConfusionMatrixDisplay
    HAS_SKLEARN = True
except Exception:
    ConfusionMatrixDisplay = None
    HAS_SKLEARN = False

pd.set_option('display.max_columns', 300)
pd.set_option('display.width', 220)


In [18]:
# Use either a run directory or a specific .pt checkpoint
MODEL_PATH = Path('src/models/saved_weights/irmas/irmas_single_label_mel_densenet_121')

# Runtime controls
BATCH_SIZE = 16
NUM_WORKERS = 0
TOP_K = 3
MAX_SAMPLES = None  # set small int (e.g. 64) for quick debug runs
DEVICE = 'auto'     # auto | cpu | cuda | mps

# Optional override. Leave as None to read from audio_params.yaml
TEST_ROOT_OVERRIDE = None



In [ ]:
result = evaluate_irmas_part1_run(
    MODEL_PATH,
    test_root_override=TEST_ROOT_OVERRIDE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    top_k=TOP_K,
    max_samples=MAX_SAMPLES,
    device=DEVICE,
    show_progress=True,
)

summary = result['summary']
summary_df = result['summary_df']
predictions_df = result['predictions_df']
per_class_df = result['per_class_df']
confusion_df = result['confusion_df']

display(summary_df)
print(f"Samples evaluated: {len(predictions_df)}")
print(f"Model checkpoint: {summary['checkpoint_path']}")
print(f"Test root: {summary['test_root']}")
print(f"Top-1 any-label accuracy: {summary['any_label_top1_acc']:.4f}")
print(f"Top-{TOP_K} any-label accuracy: {summary[f'any_label_top{TOP_K}_acc']:.4f}")
print(f"Label pickup accuracy top-1: {summary['label_pickup_acc_top1']:.4f}")
print(f"Label pickup accuracy top-{TOP_K}: {summary[f'label_pickup_acc_top{TOP_K}']:.4f}")
print(f"Top-1 macro F1 (multi-label view): {summary['top1_macro_f1']:.4f}")
print(f"Top-1 micro F1 (multi-label view): {summary['top1_micro_f1']:.4f}")



In [ ]:
display(per_class_df.sort_values('class_name').reset_index(drop=True))

# Highest-confidence mistakes (top-1 misses)
misses = predictions_df[predictions_df['top1_hit_any'] == False].copy()
display(
    misses.sort_values('pred_top1_conf', ascending=False)
    .head(25)
    .reset_index(drop=True)
)


In [ ]:
# Confusion matrix over all true labels (each true label contributes one count)
display(confusion_df)

cm = confusion_df.to_numpy(dtype=float)
labels = confusion_df.columns.tolist()
cm_norm = cm.copy()
for i in range(cm_norm.shape[0]):
    row_sum = cm_norm[i].sum()
    if row_sum > 0:
        cm_norm[i] = cm_norm[i] / row_sum

if HAS_SKLEARN:
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    disp_counts = ConfusionMatrixDisplay(confusion_matrix=cm.astype(int), display_labels=labels)
    disp_counts.plot(ax=axes[0], cmap='Blues', colorbar=False, xticks_rotation=90, values_format='d')
    axes[0].set_title('Confusion Matrix (Counts)')

    disp_norm = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=labels)
    disp_norm.plot(ax=axes[1], cmap='Blues', colorbar=False, xticks_rotation=90, values_format='.2f')
    axes[1].set_title('Confusion Matrix (Row-Normalized)')

    plt.tight_layout()
    plt.show()
else:
    print('scikit-learn is not available; showing matplotlib heatmaps instead.')
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    im0 = axes[0].imshow(cm, cmap='Blues', aspect='auto')
    axes[0].set_title('Confusion Matrix (Counts)')
    axes[0].set_xticks(range(len(labels)))
    axes[0].set_yticks(range(len(labels)))
    axes[0].set_xticklabels(labels, rotation=90)
    axes[0].set_yticklabels(labels)
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('True')
    fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

    im1 = axes[1].imshow(cm_norm, cmap='Blues', aspect='auto', vmin=0.0, vmax=1.0)
    axes[1].set_title('Confusion Matrix (Row-Normalized)')
    axes[1].set_xticks(range(len(labels)))
    axes[1].set_yticks(range(len(labels)))
    axes[1].set_xticklabels(labels, rotation=90)
    axes[1].set_yticklabels(labels)
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('True')
    fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()
